# N-CMAPSS DS05 — decomposing the gap to MSTSDN

## What v6 ruled out

The learning curve was flat, not descending: 9.423 -> 9.722 -> 8.884 across 292k, 731k and 1.46M
training rows, every pair within noise and not even monotonic. My printed verdict compared only the
endpoints against a mean standard deviation and was wrong. The paper confirms it independently —
**MSTSDN samples one point in every 50, so it trains on five times less data than we do.**

The unit sweep also showed validation swinging from 5.96 to 10.46 depending only on which engine
is held out, while test moved the other way. Another reason the objective here is unreliable.

## What the paper actually says (sections 4.2.2-4.2.3)

Three differences we never matched:

| | MSTSDN | ours |
|---|---|---|
| RUL threshold | **capped at 55** | uncapped |
| sensors | **X_s (14 physical) + X_v (14 virtual) = 28** | W (4 conditions) + X_s = 18 |
| input | **sliding window, T = 30** | single snapshot, no time axis |
| subsample | 1-in-50 | 1-in-10 |

The cap matters most for the metric: 28.7% of DS05 test rows sit above 55, and capping moves the
trivial floor from 23.36 to 18.46. Our 8.247 and their 3.95 are not measured on the same target.

The virtual sensors are model-derived internal states rather than physical measurements, which is
worth a sentence in the paper — they are not available from a real engine.

## This notebook

A **2 x 2 x 2 factorial** over cap, sensor set and window, at their 1-in-50 sampling so every cell
is comparable and cheap. The architecture stays our parallel CNN-LSTM throughout, so the factorial
measures how much of the 4.3 gap is **protocol**. Whatever remains is architecture — and MSTSDN's is
a pyramidal multiscale temporal transformer plus a multi-branch graph convolutional network, which
is a separate build, not a variation of ours.

Every configuration is scored on **both** target conventions — against the raw RUL and against the
capped-55 target with clipped predictions — so all eight cells can be read on MSTSDN's metric as
well as on ours.

In [ ]:
import os, json, gc, time, random, math, itertools
os.environ['PYTHONHASHSEED']='42'
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

In [ ]:
SEED=42; SUB=50; VAL_UNITS=[5,6]; N_SEEDS=3
BATCH=512; EPOCHS=100; PATIENCE=15; CAP=55; T=30
set_seeds(SEED)
def read_sub(dset,step,dtype=np.float32,chunk=400_000):
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=dset[a:b]; first=(-a)%step
        if first<len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out)
with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Wd=read_sub(h['W_dev'],SUB);  Sd=read_sub(h['X_s_dev'],SUB); Vd=read_sub(h['X_v_dev'],SUB)
    Ad=read_sub(h['A_dev'],SUB);  Yd=read_sub(h['Y_dev'],SUB)
    Wt=read_sub(h['W_test'],SUB); St=read_sub(h['X_s_test'],SUB); Vt=read_sub(h['X_v_test'],SUB)
    At=read_sub(h['A_test'],SUB); Yt=read_sub(h['Y_test'],SUB)
ud,cd,yd=Ad[:,0].astype(int),Ad[:,1].astype(int),Yd.ravel().astype(np.float32)
ut,ct,yt=At[:,0].astype(int),At[:,1].astype(int),Yt.ravel().astype(np.float32)
FEATSETS={'ours (W+X_s, 18)':(np.hstack([Wd,Sd]),np.hstack([Wt,St])),
          'MSTSDN (X_s+X_v, 28)':(np.hstack([Sd,Vd]),np.hstack([St,Vt]))}
del Wd,Sd,Vd,Ad,Yd,Wt,St,Vt,At,Yt; gc.collect()
print('1-in-%d sampling: dev %s rows | test %s rows'%(SUB,format(len(yd),','),format(len(yt),',')))
print('rows per flight after sampling: ~%d'%(len(yd)//len(set(ud*1000+cd))))
TR_U=[u for u in sorted(set(ud)) if u not in VAL_UNITS]

def windows(F,u,c,y,mask,T):
    """T consecutive sampled points inside one flight; label = RUL of that flight."""
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key=FF[o],yy[o],key[o]
    bnd=np.flatnonzero(np.diff(key))+1
    X,Y=[],[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(T,len(A)+1):
            X.append(A[e-T:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

def build_data(featset,use_window):
    Fd,Ft=FEATSETS[featset]; NF=Fd.shape[1]
    tr=~np.isin(ud,VAL_UNITS); va=np.isin(ud,VAL_UNITS); te=np.ones(len(yt),bool)
    sc=MinMaxScaler().fit(Fd[tr])
    Fd2=sc.transform(Fd).astype(np.float32); Ft2=sc.transform(Ft).astype(np.float32)
    if use_window:
        Xtr,ytr=windows(Fd2,ud,cd,yd,tr,T); Xva,yva=windows(Fd2,ud,cd,yd,va,T)
        Xte,yte=windows(Ft2,ut,ct,yt,te,T); shape=(T,NF)
    else:
        Xtr,ytr=Fd2[tr].reshape(-1,NF,1),yd[tr]; Xva,yva=Fd2[va].reshape(-1,NF,1),yd[va]
        Xte,yte=Ft2.reshape(-1,NF,1),yt; shape=(NF,1)
    return dict(Xtr=Xtr,ytr=ytr,Xva=Xva,yva=yva,Xte=Xte,yte=yte,shape=shape)

for fs in FEATSETS:
    for w in (False,True):
        d=build_data(fs,w)
        print('%-22s window=%-5s shape %-10s train %8s | val %7s | test %7s'
              %(fs,w,str(d['shape']),format(len(d['ytr']),','),format(len(d['yva']),','),
                format(len(d['yte']),',')))
        del d; gc.collect()

In [ ]:
def build(shape,seed=SEED,filters=(96,128,128,32),lstm_units=80,dense=(192,96),
          dropout=0.1,l2reg=0.0,lr=1e-3):
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
            kernel_regularizer=regularizers.l2(l2reg) if l2reg>0 else None)
    inp=Input(shape=shape); c=inp
    for i,f in enumerate(filters):
        c=Conv1D(f,3,padding='same',activation='relu',**kw)(c)
        if dropout>0: c=Dropout(dropout,seed=seed)(c)
        if i<2 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    cv=GlobalAveragePooling1D()(c)
    r=LSTM(lstm_units,**kw)(inp)
    h=Concatenate()([cv,r])
    for dd in dense: h=Dense(dd,activation='relu',**kw)(h)
    m=Model(inp,Dense(1,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

def run(d,cap,seed):
    ytr=np.minimum(d['ytr'],cap) if cap else d['ytr']
    yva=np.minimum(d['yva'],cap) if cap else d['yva']
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    with tf.device(DEVICE):
        m=build(d['shape'],seed=seed)
        m.fit(d['Xtr'],ytr,validation_data=(d['Xva'],yva),epochs=EPOCHS,batch_size=BATCH,
              verbose=0,callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                                   ReduceLROnPlateau('val_loss',factor=0.5,patience=PATIENCE//2,min_lr=1e-6)])
        p=m.predict(d['Xte'],batch_size=8192,verbose=0).ravel()
        pv=m.predict(d['Xva'],batch_size=8192,verbose=0).ravel()
    raw=float(np.sqrt(mean_squared_error(d['yte'],p)))                       # against uncapped RUL
    capped=float(np.sqrt(mean_squared_error(np.minimum(d['yte'],CAP),np.clip(p,0,CAP))))
    val=float(np.sqrt(mean_squared_error(yva,pv)))
    return dict(raw=raw,capped=capped,val=val,params=int(m.count_params()),secs=round(time.time()-t0))

triv_raw=float(np.sqrt(mean_squared_error(yt,np.full_like(yt,yd[~np.isin(ud,VAL_UNITS)].mean()))))
ycap=np.minimum(yt,CAP); dcap=np.minimum(yd[~np.isin(ud,VAL_UNITS)],CAP)
triv_cap=float(np.sqrt(mean_squared_error(ycap,np.full_like(ycap,dcap.mean()))))
print('trivial floors -> raw target %.3f | capped-%d target %.3f'%(triv_raw,CAP,triv_cap))

## The factorial — cap x sensors x window

In [ ]:
RES=[]; t0=time.time()
for fs,wnd,cap in itertools.product(list(FEATSETS),(False,True),(None,CAP)):
    d=build_data(fs,wnd)
    rs=[run(d,cap,SEED+s) for s in range(N_SEEDS)]
    raw=np.array([r['raw'] for r in rs]); cp=np.array([r['capped'] for r in rs])
    RES.append(dict(sensors='28' if 'MSTSDN' in fs else '18',window='T=30' if wnd else 'none',
                    cap=CAP if cap else 'none',raw=raw.mean(),raw_sd=raw.std(ddof=1),
                    capped=cp.mean(),capped_sd=cp.std(ddof=1),
                    val=np.mean([r['val'] for r in rs]),n_train=len(d['ytr']),
                    secs=np.mean([r['secs'] for r in rs])))
    print('sensors=%-3s window=%-5s cap=%-5s | raw %6.3f +/- %5.3f | capped %6.3f +/- %5.3f | %4.0fs'
          %(RES[-1]['sensors'],RES[-1]['window'],str(RES[-1]['cap']),
            raw.mean(),raw.std(ddof=1),cp.mean(),cp.std(ddof=1),RES[-1]['secs']))
    del d; gc.collect()
F=pd.DataFrame(RES); pd.set_option('display.width',220)
print('\n'+F[['sensors','window','cap','raw','raw_sd','capped','capped_sd','val','n_train']].to_string(index=False))
F.to_csv('ncmapss_protocol_factorial.csv',index=False)
print('\nfactorial finished in %.0f s'%(time.time()-t0))

## Main effects — how much of the gap is protocol?

In [ ]:
print('=== main effect of each factor, on the CAPPED metric (MSTSDN\'s convention) ===')
for col,a,b in [('sensors','18','28'),('window','none','T=30'),('cap','none',CAP)]:
    ma=F[F[col].astype(str)==str(a)].capped.mean(); mb=F[F[col].astype(str)==str(b)].capped.mean()
    print('  %-8s %-6s %6.3f  ->  %-6s %6.3f   effect %+.3f'%(col,str(a),ma,str(b),mb,mb-ma))
print()
print('=== main effect on the RAW metric (our convention) ===')
for col,a,b in [('sensors','18','28'),('window','none','T=30')]:
    ma=F[F[col].astype(str)==str(a)].raw.mean(); mb=F[F[col].astype(str)==str(b)].raw.mean()
    print('  %-8s %-6s %6.3f  ->  %-6s %6.3f   effect %+.3f'%(col,str(a),ma,str(b),mb,mb-ma))
print()
ours=F[(F.sensors=='18')&(F.window=='none')&(F.cap=='none')].iloc[0]
mst =F[(F.sensors=='28')&(F.window=='T=30')&(F.cap==CAP)].iloc[0]
print('='*76)
print('%-46s %8s %8s'%('','raw','capped'))
print('%-46s %8.3f %8.3f'%('our original protocol (18, no window, no cap)',ours.raw,ours.capped))
print('%-46s %8.3f %8.3f'%('MSTSDN protocol (28, T=30, cap 55)',mst.raw,mst.capped))
print('%-46s %8s %8.2f'%('MSTSDN published',' -- ',3.95))
print('%-46s %8.3f %8.3f'%('trivial floor',triv_raw,triv_cap))
print('='*76)
closed=ours.capped-mst.capped; remain=mst.capped-3.95
print('\nprotocol alone closes %+.3f of the gap'%(-closed))
print('remaining gap to MSTSDN on their own metric: %+.3f'%remain)
print('\nthat remainder is ARCHITECTURE: MSTSDN is a pyramidal multiscale temporal transformer')
print('plus a multi-branch graph convolutional network, not a variant of a CNN-LSTM.')
json.dump(dict(factorial=RES,trivial_raw=triv_raw,trivial_capped=triv_cap,
               ours=dict(ours),mstsdn_protocol=dict(mst),published=3.95),
          open('ncmapss_protocol_factorial.json','w'),indent=2,default=str)
print('\nsaved ncmapss_protocol_factorial.csv and .json')